In [10]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()


hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

print("Active SparkSession:", spark.sparkContext.uiWebUrl)

Active SparkSession: http://3136f3ad71e9:4040


In [2]:
# ⬇️ Параметры подключения к PostgreSQL 
# public.shops 
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shops" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shops_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

shops_df.show()

+-----+-----------+
|st_id|  shop_name|
+-----+-----------+
|  842|      Lenta|
|  843|     Magnit|
|  844|       Spar|
|  845|Pyaterochka|
|  846|      Lenta|
|  847|      Diksi|
|  848|      Lenta|
|  849|   FixPrice|
|  850|     Magnit|
|  851|      Lenta|
+-----+-----------+



In [3]:
# public.shop_timezone
jdbc_url = "jdbc:postgresql://postgres_source:5432/source" 
table_name = "public.shop_timezone" 
db_user = os.getenv("POSTGRES_USER") 
db_password = os.getenv("POSTGRES_PASSWORD") 
shop_timezone_df = spark.read \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("user", db_user) \
    .option("password", db_password) \
    .option("dbtable", table_name) \
    .option("fetchsize", 1000) \
    .option("driver", "org.postgresql.Driver") \
    .load()

shop_timezone_df.show()

+-----+---------+
|plant|time_zone|
+-----+---------+
|  842|         |
|  842|    RUS07|
|  843|    RUS04|
|  844|         |
|  845|         |
|  845|    RUS05|
|  847|    RUS03|
|  848|    RUS08|
|  848|         |
| P847|         |
| E103|    RUS08|
| -134|    RUS04|
|    0|         |
|    0|    RUS08|
|  848|         |
+-----+---------+



In [19]:
shops_df.printSchema()
shop_timezone_df.printSchema()

shops_df.columns + shop_timezone_df.columns

shops_df.explain("cost")
shop_timezone_df.explain("cost")


root
 |-- st_id: integer (nullable = true)
 |-- shop_name: string (nullable = true)

root
 |-- plant: string (nullable = true)
 |-- time_zone: string (nullable = true)

== Optimized Logical Plan ==
Relation [st_id#0,shop_name#1] JDBCRelation(public.shops) [numPartitions=1], Statistics(sizeInBytes=8.0 EiB)

== Physical Plan ==
*(1) Scan JDBCRelation(public.shops) [numPartitions=1] [st_id#0,shop_name#1] PushedFilters: [], ReadSchema: struct<st_id:int,shop_name:string>


== Optimized Logical Plan ==
Relation [plant#13,time_zone#14] JDBCRelation(public.shop_timezone) [numPartitions=1], Statistics(sizeInBytes=8.0 EiB)

== Physical Plan ==
*(1) Scan JDBCRelation(public.shop_timezone) [numPartitions=1] [plant#13,time_zone#14] PushedFilters: [], ReadSchema: struct<plant:string,time_zone:string>




In [4]:
# Регистрируем DataFrame-ы как временные вьюхи 
shops_df.createOrReplaceTempView("shops") 
shop_timezone_df.createOrReplaceTempView("shop_timezone")

In [74]:

spark.sql("""
WITH st AS (SELECT plant, 
CASE
    WHEN time_zone LIKE 'RUS0%' THEN REGEXP_REPLACE (time_zone, 'RUS0', '')
    ELSE 3
    END tz_code
FROM shop_timezone
)

SELECT st_id, shop_name, tz_code
FROM (SELECT st_id, shop_name, tz_code,
    ROW_NUMBER() OVER (PARTITION BY st_id ORDER BY (CASE WHEN st.tz_code = '3' THEN 2 ELSE 1 END)) rn
    FROM shops s
    LEFT JOIN st ON s.st_id = st.plant
    WHERE tz_code IS NOT NULL) as t
WHERE rn = 1

""").show()


+-----+-----------+-------+
|st_id|  shop_name|tz_code|
+-----+-----------+-------+
|  842|      Lenta|      7|
|  843|     Magnit|      4|
|  844|       Spar|      3|
|  845|Pyaterochka|      5|
|  847|      Diksi|      3|
|  848|      Lenta|      8|
+-----+-----------+-------+



In [75]:
spark.stop()

In [20]:
spark